# Alexandria End-to-End Local Walkthrough

This notebook mirrors `tests/integration/test_end_to_end_flow.py`. It uses an in-memory SQLite database plus deterministic fake provider ports, so no OpenAI credentials, Postgres service, or live network calls are required.

In [ ]:
from collections.abc import Callable
from uuid import UUID

from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session, sessionmaker
from sqlalchemy.pool import StaticPool

from application.app import App
import application.app as app_module
from application.ports import ChildPlan, DocIn, SplitPlan
from domain.entity import Base, Document, Job, Node, VECTOR_DIMENSIONS
from domain.values import JobKind, JobStatus
from infrastructure.config import IngestSettings, Settings
from presentation.worker.app import Worker

In [ ]:
class MemoryDb:
    def __init__(self) -> None:
        self._engine = create_engine(
            "sqlite+pysqlite://",
            connect_args={"check_same_thread": False},
            poolclass=StaticPool,
        )
        self._sessions = sessionmaker(bind=self._engine)

    def sessions(self) -> sessionmaker[Session]:
        return self._sessions

    def session(self) -> Session:
        return self._sessions()

    def create_all(self) -> None:
        Base.metadata.create_all(self._engine)

    def close(self) -> None:
        self._engine.dispose()


def vector(*values: float) -> list[float]:
    embedding = [0.0] * VECTOR_DIMENSIONS
    for index, value in enumerate(values):
        embedding[index] = value
    return embedding


def deterministic_embedding(text: str) -> list[float]:
    lowered = text.lower()
    if "beta" in lowered:
        return vector(0.0, 1.0)
    if "alpha" in lowered:
        return vector(1.0, 0.0)
    return vector(0.0, 0.0, 1.0)

In [ ]:
class FakeEmbedder:
    def __init__(self, embed: Callable[[str], list[float]]) -> None:
        self.embed_text = embed
        self.calls: list[str] = []

    async def embed(self, text: str) -> list[float]:
        self.calls.append(text)
        return self.embed_text(text)


class FakeSummarizer:
    def __init__(self) -> None:
        self.calls: list[DocIn] = []

    async def summarize(self, doc: DocIn) -> str:
        self.calls.append(doc)
        return f"Summary for {doc.name}."


class DeterministicSplitter:
    def __init__(self) -> None:
        self.calls: list[tuple[UUID, list[UUID]]] = []

    async def split(self, node: Node, docs: list[Document]) -> SplitPlan:
        ordered = sorted(docs, key=lambda item: item.name)
        self.calls.append((node.id, [doc.id for doc in ordered]))
        return SplitPlan(
            children=[
                ChildPlan(
                    name="Alpha lifecycle",
                    description="Documents about alpha lifecycle work.",
                    embedding=vector(1.0, 0.0),
                    docs=[ordered[0].id, ordered[2].id],
                ),
                ChildPlan(
                    name="Beta operations",
                    description="Documents about beta operations.",
                    embedding=vector(0.0, 1.0),
                    docs=[ordered[1].id],
                ),
            ]
        )

In [ ]:
db = MemoryDb()
embedder = FakeEmbedder(deterministic_embedding)
summarizer = FakeSummarizer()
splitter = DeterministicSplitter()

app_module.Db = lambda _settings: db
app_module.make_embedder = lambda _provider, _settings: embedder
app_module.make_summarizer = lambda _provider, _settings: summarizer

settings = Settings(_env_file=None, ingest=IngestSettings(max_leaf_docs=3))
app = App(settings, splitter=splitter)

In [ ]:
docs = [
    DocIn(
        name="Alpha",
        body="Alpha local lifecycle notes about setup, routing, and retrieval.",
        source_key="source:alpha",
    ),
    DocIn(
        name="Beta",
        body="Beta operations note about queue workers and retries.",
        source_key="source:beta",
    ),
    DocIn(
        name="Gamma",
        body="Gamma archive note for the split lifecycle.",
        source_key="source:gamma",
    ),
]

root = await app.seed()
ids = [await app.ingest(doc) for doc in docs]
before_split = await app.retrieve("alpha local lifecycle", limit=3)
processed = await Worker(app=app, kind=JobKind.SPLIT_CHECK).batch()
app.session.expire_all()
after_split = await app.retrieve("alpha local lifecycle", limit=1)

print("root", root.id)
print("document ids", ids)
print("before split", [hit.doc.source_key for hit in before_split])
print("worker processed", processed)
print("after split", [hit.doc.source_key for hit in after_split])

In [ ]:
with db.session() as session:
    saved_root = session.scalar(select(Node).where(Node.parent_id.is_(None)))
    children = session.scalars(
        select(Node).where(Node.parent_id == saved_root.id).order_by(Node.name.asc())
    ).all()
    job = session.scalar(select(Job).where(Job.kind == JobKind.SPLIT_CHECK))

assert saved_root.kind == "branch"
assert [(child.name, child.doc_count) for child in children] == [
    ("Alpha lifecycle", 2),
    ("Beta operations", 1),
]
assert job.status == JobStatus.DONE.value
assert [hit.doc.source_key for hit in before_split][0] == "source:alpha"
assert [hit.doc.source_key for hit in after_split] == ["source:alpha"]

app.close()

The automated version of this walkthrough is:

```bash
uv run pytest tests/integration/test_end_to_end_flow.py -q
```